# Global RF
- Preprocess data
- Compare linear and non linear model
- Explore Clustering of residuals
- Spatial Crosvalidation

In [ ]:
import pathlib

import esda
import geopandas as gpd
import joblib
import matplotlib.patheffects as patheffects
import matplotlib.pyplot as plt
import matplotlib.transforms as transforms
import numpy as np
import osmnx as ox
import pandas as pd
import pyinterpolate
import seaborn as sns
import shapely
from libpysal import graph
from matplotlib.colors import ListedColormap
from matplotlib.legend_handler import HandlerBase
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Path, PathPatch
from matplotlib.transforms import Bbox
from matplotlib_map_utils.core.north_arrow import NorthArrow
from matplotlib_scalebar.scalebar import ScaleBar
from mpl_toolkits.axes_grid1 import inset_locator
from shapely import wkb
from shapely.ops import linemerge, polygonize, unary_union
from sklearn import ensemble, metrics, model_selection
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import StandardScaler

## Preprocess the data
Remove unnecessary column, devide the data by total population and standardize it.

In [ ]:
def process_file(path, path_total):
    cols_no_division = [
        "Hustota obyvatel na obytnou plochu",
        "Počet obyvatel na dům",
        "Počet obyvatel na byt",
    ]

    # Load total population data
    total = pd.read_csv(path_total, dtype={"nadzsjd": str}, index_col=0)

    # Load main data
    data = gpd.read_parquet(path)

    # Merge data
    data_total = data.join(total)

    # Drop unnecessary columns
    data_relative = data_total.drop(
        columns=[
            "NUTS_2",
            "naz_oblast",
            "NUTS_3",
            "naz_kraj",
            #    "kod_okres",
            "naz_okres",
            "naz_orp",
            "kod_orp",
            "kod_obec",
            "naz_obec",
            "kod_mco",
            "nazev_mco",
        ]
    )

    # Convert columns (except 'geometry') to float
    cols_numeric = data_relative.columns.drop(["geometry", "kod_okres"])
    data_relative[cols_numeric] = data_relative[cols_numeric].astype(float)

    # Normalize by total population (except certain columns)
    population = data_relative["Obyvatelstvo celkem"].replace(
        0, np.nan
    )  # avoid division by zero
    cols_to_normalize = [
        col
        for col in cols_numeric
        if col not in cols_no_division + ["Obyvatelstvo celkem"]
    ]
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].div(
        population, axis=0
    )

    # Drop rows with NaNs after division
    data_relative = data_relative.dropna(subset=cols_to_normalize)

    # Clip extremely large or small values
    data_relative[cols_to_normalize] = data_relative[cols_to_normalize].clip(-1e6, 1e6)
    data_relative[cols_no_division] = data_relative[cols_no_division].clip(-1e6, 1e6)

    # Replace any remaining infinities with NaN and drop them
    data_relative.replace([np.inf, -np.inf], np.nan, inplace=True)
    data_relative.dropna(subset=cols_to_normalize + cols_no_division, inplace=True)

    # Scale columns
    scaler = StandardScaler()
    data_relative[cols_to_normalize] = scaler.fit_transform(
        data_relative[cols_to_normalize]
    )
    data_relative[cols_no_division] = scaler.fit_transform(
        data_relative[cols_no_division]
    )

    return data_relative

In [ ]:
path = "/data/uscuni-restricted/04_spatial_census/_merged_census_2021.parquet"
path_total = "/data/uscuni-restricted/04_spatial_census/total.csv"

In [ ]:
data_relative = process_file(path, path_total)

In [ ]:
# data_r = data_relative[data_relative.columns.drop(["Obyvatelstvo celkem", "kod_okres"])]
# data_r.to_parquet(
#    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
# )

## Assign cluster label


In [ ]:
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v11.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = data_relative.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(
    [
        "geometry",
        "kod_nadzsj_d",
        "final_without_noise",
        "kod_okres",
        "Obyvatelstvo celkem",
    ]
)

# data["Cluster"] = data["final_without_noise"].map(cluster_mapping[3])
data["Cluster"] = data["final_without_noise"]

In [ ]:
data

In [ ]:
fa = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní",
    "Počet obyvatel na byt",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Obyvatelstvo - věk: 0 - 6  - celkem",
    "Obyvatelstvo - věk: 7 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s trvalým pobytem - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
]

## Classification


In [ ]:
# Assign independent variables and the target
independent = data[fa]
# independent = data.drop(columns=["geometry","kod_okres","kod_nadzsj_d","Obyvatelstvo celkem","final_without_noise", "Cluster"])
target = data["Cluster"]
# target = (data["Cluster"] == 5).astype(int)

In [ ]:
cluster_colors = {
    1: "#4069BC",
    2: "#7CBAE4",
    3: "#E69C63",
    4: "#eec1d5",
    5: "#E0665F",
    6: "#ECBF43",
    7: "#b2cd32",
    8: "#1F943E",
}
data["cluster_color"] = data["Cluster"].map(cluster_colors)

In [ ]:
# Plot original data
ax = data.plot(
    color=data["cluster_color"],
    legend=True,
    figsize=(9, 9),
    markersize=0.1,
    categorical=True,
)
ax.set_axis_off()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    independent,
    target,
    test_size=0.2,
    random_state=42,
)

### Linear model


In [ ]:
linear_model = LogisticRegression(
    n_jobs=-1, max_iter=5000, solver="saga", random_state=42, class_weight="balanced"
)
linear_model.fit(X_train, y_train)
linear_model.score(X_train, y_train), linear_model.score(X_test, y_test)

In [ ]:
y_pred_train = linear_model.predict(X_train)
y_pred_test = linear_model.predict(X_test)

f1_train = f1_score(y_train, y_pred_train, average="macro")
f1_test = f1_score(y_test, y_pred_test, average="macro")

f1_train, f1_test

### Non-Linear model


In [ ]:
rf_model = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")
rf_model.fit(X_train, y_train)
rf_model.score(X_train, y_train), rf_model.score(X_test, y_test)

#### Parameter tuning


In [ ]:
np.mean([estimator.tree_.max_depth for estimator in rf_model.estimators_])

In [ ]:
rf_model = ensemble.RandomForestClassifier(
    min_samples_split=10,
    min_samples_leaf=5,
    max_depth=15,
    n_jobs=-1,
    random_state=42,
    n_estimators=300,
    max_features="log2",
    class_weight="balanced",
)
rf_model.fit(X_train, y_train)

In [ ]:
rf_model.score(X_train, y_train), rf_model.score(X_test, y_test)

In [ ]:
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

f1_train = f1_score(y_train, y_pred_train, average="macro")
f1_test = f1_score(y_test, y_pred_test, average="macro")

f1_train, f1_test

In [ ]:
pred = rf_model.predict(X_test)

In [ ]:
accuracy = metrics.accuracy_score(pred, y_test)
kappa = metrics.cohen_kappa_score(pred, y_test)

summary = f"""\
Evaluation metrics
==================
Basic model:
  Accuracy: {round(accuracy, 3)}
  Kappa:    {round(kappa, 3)}
"""

print(summary)

In [ ]:
predicted = model_selection.cross_val_predict(
    rf_model, independent, target, cv=4, n_jobs=-1
)

ax = data.plot(predicted, legend=True, figsize=(9, 9), markersize=0.1, categorical=True)
ax.set_axis_off()

In [ ]:
contiguity = graph.Graph.build_knn(data.centroid, k=30)
contiguity_b = contiguity.transform("b")

In [ ]:
for variable in independent:
    gs_global = esda.Moran(data[variable], contiguity_b)
    print(
        f"Moran's I of {variable} is {gs_global.I:.2f} with the  p-value of {gs_global.p_sim}."
    )

In [ ]:
for variable in independent:
    gs_global = esda.G(data[variable], contiguity_b)
    print(
        f"Global G of {variable} is {gs_global.G:.10f} with the  p-value of {gs_global.p_sim}."
    )

### Spatial Crossvalidation

In [ ]:
gkf = model_selection.StratifiedGroupKFold(n_splits=5)
splits = gkf.split(
    independent,
    target,
    groups=data.kod_okres,
)

In [ ]:
split_label = np.empty(len(data), dtype=float)
split_label

In [ ]:
for i, (_train_idx, test_idx) in enumerate(splits):
    split_label[test_idx] = i
data["split"] = split_label

In [ ]:
ax = data.plot("split", categorical=True, figsize=(9, 9), markersize=0.1, legend=True)
data.dissolve("kod_okres").convex_hull.boundary.plot(
    ax=ax, color="k", linewidth=0.5, markersize=0
)
ax.set_axis_off()

In [ ]:
train = data["split"] != 0
X_train = independent.loc[train]
y_train = data["Cluster"].loc[train]

test = data["split"] == 0
X_test = independent.loc[test]
y_test = data["Cluster"].loc[test]

In [ ]:
rf_spatial_cv = rf_model
rf_spatial_cv.fit(X_train, y_train)

In [ ]:
rf_spatial_cv.score(X_train, y_train), rf_spatial_cv.score(X_test, y_test)

In [ ]:
pred = rf_spatial_cv.predict(X_test)

accuracy_spatial_cv = metrics.accuracy_score(pred, y_test)
kappa_spatial_cv = metrics.cohen_kappa_score(pred, y_test)

summary += f"""\
Basic model with spatial cross-validation:
  Accuracy: {round(accuracy_spatial_cv, 3)}
  Kappa:    {round(kappa_spatial_cv, 3)}
"""

print(summary)

In [ ]:
report_dict = classification_report(y_test, pred, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

report_df

In [ ]:
# NON-LINEAR
predicted_spatial_rf = model_selection.cross_val_predict(
    rf_spatial_cv, independent, target, cv=5, n_jobs=-1
)

In [ ]:
report_dict = classification_report(target, predicted_spatial_rf, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

report_df = report_df.rename(
    index={
        "1": "Incoherent Large-Scale Homogeneous Fabric",
        "2": "Incoherent Large-Scale Heterogeneous Fabric",
        "3": "Incoherent Small-Scale Linear Fabric",
        "4": "Incoherent Small-Scale Sparse Fabric",
        "5": "Incoherent Small-Scale Compact Fabric",
        "6": "Coherent Interconnected Fabric",
        "7": "Coherent Dense Disjoint Fabric",
        "8": "Coherent Dense Adjacent Fabric",
    }
)
report_df.round(2)

In [ ]:
# LINEAR
predicted_spatial_lr = model_selection.cross_val_predict(
    linear_model, independent, target, cv=5, n_jobs=-1
)

In [ ]:
report_dict = classification_report(target, predicted_spatial_lr, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()
report_df = report_df.rename(
    index={
        "1": "Incoherent Large-Scale Homogeneous Fabric",
        "2": "Incoherent Large-Scale Heterogeneous Fabric",
        "3": "Incoherent Small-Scale Linear Fabric",
        "4": "Incoherent Small-Scale Sparse Fabric",
        "5": "Incoherent Small-Scale Compact Fabric",
        "6": "Coherent Interconnected Fabric",
        "7": "Coherent Dense Disjoint Fabric",
        "8": "Coherent Dense Adjacent Fabric",
    }
)
report_df.round(2)

### Residual exploration

In [ ]:
data["error_rf"] = predicted_spatial_rf == target
data["error_rf"] = data["error_rf"].astype(int).astype(float).values
data["error_lr"] = predicted_spatial_lr == target
data["error_lr"] = data["error_lr"].astype(int).astype(float).values

In [ ]:
contiguity = graph.Graph.build_knn(data.centroid, k=90)
contiguity_b = contiguity.transform("b")

### Join Counts

In [ ]:
jc_rf = esda.Join_Counts(data["error_rf"], contiguity_b)


# Results
print("BB count:", jc_rf.bb)  # 1–1 joins
print("WW count:", jc_rf.ww)  # 0–0 joins
print("BW count:", jc_rf.bw)
print("mean BB:", jc_rf.mean_bb)
print("p-value:", jc_rf.p_sim_bb)

In [ ]:
jc_lr = esda.Join_Counts(data["error_lr"], contiguity_b)


# Results
print("BB count:", jc_lr.bb)  # 1–1 joins
print("WW count:", jc_lr.ww)  # 0–0 joins
print("BW count:", jc_lr.bw)
print("mean BB:", jc_lr.mean_bb)
print("p-value:", jc_lr.p_sim_bb)

#### Getirs Ord

In [ ]:
g_global_rf = esda.G(data["error_rf"], contiguity_b, permutations=999)
print(g_global_rf.G)  # global G statistic
print(g_global_rf.p_sim)
g_global_lr = esda.G(data["error_lr"], contiguity_b, permutations=999)
print(g_global_lr.G)  # global G statistic
print(g_global_lr.p_sim)

In [ ]:
g_local_rf = esda.G_Local(data["error_rf"], contiguity_b, star=True, permutations=999)
data["G_sig_rf"] = g_local_rf.p_sim < 0.05

g_local_lr = esda.G_Local(data["error_lr"], contiguity_b, star=True, permutations=999)
data["G_sig_lr"] = g_local_lr.p_sim < 0.05

In [ ]:
data["spot_type_rf"] = "Insignificant"

data.loc[(g_local_rf.p_sim < 0.05) & (g_local_rf.Zs > 0), "spot_type_rf"] = "Hot Spot"
data.loc[(g_local_rf.p_sim < 0.05) & (g_local_lr.Zs < 0), "spot_type_rf"] = "Cold Spot"

data["spot_type_rf"].value_counts()

In [ ]:
data["spot_type_lr"] = "Insignificant"

data.loc[(g_local_lr.p_sim < 0.05) & (g_local_rf.Zs > 0), "spot_type_lr"] = "Hot Spot"
data.loc[(g_local_lr.p_sim < 0.05) & (g_local_lr.Zs < 0), "spot_type_lr"] = "Cold Spot"

data["spot_type_lr"].value_counts()

In [ ]:
my_colors = ["#DDDCDC", "#4069BC"]
my_cmap = ListedColormap(my_colors)

In [ ]:
text_outline = [patheffects.withStroke(linewidth=3, foreground="white")]

In [ ]:
sns.set_context("paper", font_scale=1.5)

In [ ]:
kraje = gpd.read_file("/data/uscuni-kindergartens/geometrie/kraje.geojson").to_crs(
    data.crs
)
stat = kraje.dissolve()

admin_level_9 = ox.features_from_place("Brno", {"admin_level": "8"})
brno = admin_level_9[admin_level_9.geom_type == "Polygon"][["name", "geometry"]]
brno = brno.to_crs(kraje.crs)

gdf_poly = gpd.read_parquet("/data/uscuni-restricted/geometries/CMS.parquet")
"""
all_lines = linemerge(unary_union(gdf["geometry"]))
polygons = list(polygonize(all_lines))

# rebuild as geodataframe
gdf_poly = gpd.GeoDataFrame(geometry=polygons, crs=gdf.crs)
gdf_poly = gdf_poly.to_crs(kraje.crs)
gdf_poly= gdf_poly.loc[[1,0,4]]
gdf_poly["name"] = "Bohemia"
gdf_poly.loc[4,"name"]= gdf_poly.loc[4, "name"] = "Silesia"
gdf_poly.loc[1,"name"]= gdf_poly.loc[1, "name"] = "Moravia"
"""

In [ ]:
prg = kraje.query("NAZEV=='Hlavní město Praha'")

In [ ]:
class HandlerPolygon(HandlerBase):
    def __init__(
        self, geom, facecolor="white", edgecolor="black", linewidth=0.8, **kwargs
    ):
        super().__init__(**kwargs)
        if geom.geom_type == "MultiPolygon":
            geom = max(geom.geoms, key=lambda g: g.area)
        # Get raw coordinates
        coords = np.array(geom.exterior.coords)
        coords -= coords.min(axis=0)  # Normalize origin to 0,0
        # Calculate strict aspect ratio
        w, h = coords.max(axis=0)
        self.aspect_ratio = w / h if h != 0 else 1.0
        self.coords = coords
        self.facecolor = facecolor
        self.edgecolor = edgecolor
        self.linewidth = linewidth

    def create_artists(self, xdescent, ydescent, width, height, trans):
        # 1. Calculate the exact drawing area allowed by the legend box
        display_w = width
        display_h = height
        # 2. Adjust dimensions to strictly respect the original geometry's aspect ratio
        if display_w / display_h > self.aspect_ratio:
            # The box is wider than our shape; shrink width to fit height
            display_w = display_h * self.aspect_ratio
        else:
            # The box is taller than our shape; shrink height to fit width
            display_h = display_w / self.aspect_ratio
        # 3. Center the shape inside the allocation box
        x_offset = -xdescent + (width - display_w) / 2.0
        y_offset = -ydescent + (height - display_h) / 2.0
        # 4. Create a transform mapping the normalized coordinates directly to this box
        src_bbox = Bbox.from_bounds(
            0, 0, self.coords[:, 0].max(), self.coords[:, 1].max()
        )
        dst_bbox = Bbox.from_bounds(x_offset, y_offset, display_w, display_h)
        final_transform = transforms.BboxTransform(src_bbox, dst_bbox) + trans

        patch = PathPatch(
            Path(self.coords),
            facecolor=self.facecolor,
            edgecolor=self.edgecolor,
            linewidth=self.linewidth,
            transform=final_transform,
        )
        return [patch]

In [ ]:
prg_geom = prg.dissolve().geometry.simplify(500).iloc[0]
brno_geom = brno.dissolve().geometry.simplify(500).iloc[0]

prg_patch = Patch(facecolor="white", edgecolor="black", label="Prague")
brno_patch = Patch(facecolor="white", edgecolor="black", label="Brno")

legend_elements = [
    prg_patch,
    brno_patch,
    Patch(facecolor="#4069BC", edgecolor="none", label="Correct prediction"),
    Patch(facecolor="#DDDCDC", edgecolor="none", label="Error/Not significant"),
    Patch(facecolor="#E0665F", edgecolor="none", label="Hot spot"),
    # DDDCDC
]

handler_map = {
    prg_patch: HandlerPolygon(
        prg_geom, facecolor="white", edgecolor="black", linewidth=0.8
    ),
    brno_patch: HandlerPolygon(
        brno_geom, facecolor="white", edgecolor="black", linewidth=0.8
    ),
}


minx, miny, maxx, maxy = prg.buffer(10000).total_bounds

text_offsets = {
    "Silesia": (10000, -10000),
    "Moravia": (0, 10000),
    "Bohemia": (-25000, -30000),
}

In [ ]:
f, axes = plt.subplots(2, 2, figsize=(16, 10))
(ax1, ax2), (ax3, ax4) = axes

# hot/cold spots
for ax, col in zip([ax3, ax4], ["spot_type_lr", "spot_type_rf"], strict=False):
    data.loc[data[col] == "Insignificant"].plot(
        ax=ax, color="#DDDCDC", linewidth=0.05, edgecolor="white"
    )
    data.loc[data[col] == "Hot Spot"].plot(
        ax=ax, color="#DDDCDC", linewidth=0.05, edgecolor="white"
    )
    data.loc[data[col] == "Cold Spot"].plot(
        ax=ax, color="#E0665F", linewidth=0.05, edgecolor="white"
    )
    stat.exterior.plot(ax=ax, color="k", linewidth=0.8)
    gdf_poly.boundary.plot(ax=ax, color="k", linewidth=0.5)
    prg.dissolve().boundary.plot(ax=ax, color="k", linewidth=0.4)
    brno.dissolve().boundary.plot(ax=ax, color="k", linewidth=0.4)
    for _idx, row in gdf_poly.iterrows():
        pt = row.geometry.representative_point()
        x, y = pt.x, pt.y
        dx, dy = text_offsets.get(row["name"], (0, 0))
        ax.text(
            x + dx,
            y + dy,
            row["name"],
            ha="center",
            va="center",
            fontsize=14,
            color="black",
            path_effects=text_outline,
        )

    ax.set_axis_off()

# error maps
for ax, col in zip([ax1, ax2], ["error_lr", "error_rf"], strict=False):
    iax = inset_locator.inset_axes(
        ax,
        width=1.8,
        height=1.8,
        loc="upper right",
        borderpad=0,
        axes_kwargs={"xticks": [], "yticks": []},
    )
    data.plot(col, ax=ax, cmap=my_cmap, linewidth=0.05, edgecolor="white")
    stat.exterior.plot(ax=ax, color="k", linewidth=0.8)
    gdf_poly.boundary.plot(ax=ax, color="k", linewidth=0.5)
    brno.dissolve().boundary.plot(ax=ax, color="k", linewidth=0.4)
    prg.dissolve().boundary.plot(ax=ax, color="k", linewidth=0.4)
    data.plot(col, ax=iax, cmap=my_cmap, linewidth=0.2, edgecolor="white")
    prg.dissolve().boundary.plot(ax=iax, color="k", linewidth=1)
    iax.set_xlim(minx, maxx)
    iax.set_ylim(miny, maxy)
    inset_locator.mark_inset(ax, iax, loc1=2, loc2=4, linewidth=1)
    ax.set_axis_off()
    for _idx, row in gdf_poly.iterrows():
        pt = row.geometry.representative_point()
        x, y = pt.x, pt.y
        dx, dy = text_offsets.get(row["name"], (0, 0))
        ax.text(
            x + dx,
            y + dy,
            row["name"],
            ha="center",
            va="center",
            fontsize=14,
            color="black",
            path_effects=text_outline,
        )


ax4.add_artist(ScaleBar(1, location="lower right"))


f.legend(
    handles=legend_elements,
    handler_map=handler_map,
    loc="lower left",
    bbox_to_anchor=(-0.0, 0.41),
    frameon=False,
    fontsize=14,
    handlelength=2.5,
    handleheight=2.0,
)

f.subplots_adjust(wspace=0, hspace=0)
plt.tight_layout()
# f.savefig("cz_correlation.png", dpi=300, format="png")

### Feature importance

In [ ]:
feat_importances = pd.Series(
    rf_model.feature_importances_, index=X_train.columns
).sort_values()
plt.figure(figsize=(5, 20))
plt.axvline(x=0.01, color="red", linestyle="--", linewidth=1)


feat_importances.plot(kind="barh")

In [ ]:
data.loc[data["Cluster"] == 2].plot()